In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import wandb


: 

WARNING: Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProxyError('Cannot connect to proxy.', NewConnectionError('<pip._vendor.urllib3.connection.HTTPSConnection object at 0x7f394f822c30>: Failed to establish a new connection: [Errno -3] Temporary failure in name resolution'))': /simple/ipykernel/


In [ ]:
# Load predictions
PREDICTIONS_FILE = "./prediction_outputs/predictions_ensemble.json"  # Change this to match your model variant

def load_predictions(predictions_file):
    with open(predictions_file, "r") as f:
        return json.load(f)["predictions"]

predictions = load_predictions(PREDICTIONS_FILE)


In [ ]:
# Set up index tracking
current_index = 0

In [ ]:
def visualize_prediction(index):
    clear_output(wait=True)
    
    if index < 0 or index >= len(predictions):
        print("⚠️ Index out of range! Adjusting to valid range.")
        index = max(0, min(index, len(predictions) - 1))
    
    entry = predictions[index]
    file_name = entry["file_name"]
    x_input = torch.tensor(entry["input_image"])  # Convert stored input back to tensor
    
    print(f"🔍 Visualizing {file_name} ({index + 1}/{len(predictions)})")
    
    mean_prediction = torch.tensor(entry["outputs"].get("mean", None))
    std_dev = torch.tensor(entry["outputs"].get("std_dev", None)) if "std_dev" in entry["outputs"] else None
    
    quantile_spread = None
    if any(key.startswith("q") for key in entry["outputs"]):
        quantile_keys = sorted([key for key in entry["outputs"].keys() if key.startswith("q")])
        if len(quantile_keys) >= 2:
            lowest_quantile = quantile_keys[0]
            highest_quantile = quantile_keys[-1]
            quantile_spread = torch.tensor(entry["outputs"][highest_quantile]) - torch.tensor(entry["outputs"][lowest_quantile])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(x_input.permute(1, 2, 0))
    axes[0].set_title("Input Image")
    
    if mean_prediction is not None:
        axes[1].imshow(mean_prediction.squeeze(), cmap="viridis")
        axes[1].set_title("Mean Prediction")
    
    if std_dev is not None:
        axes[2].imshow(std_dev.squeeze(), cmap="Reds")
        axes[2].set_title("Std Dev (Uncertainty)")
    elif quantile_spread is not None:
        axes[2].imshow(quantile_spread.squeeze(), cmap="Blues")
        axes[2].set_title("Quantile Spread (Uncertainty)")
    
    plt.show()
    
    if wandb.run:
        wandb.log({
            f"{file_name}/input": wandb.Image(x_input.permute(1, 2, 0)),
            f"{file_name}/mean": wandb.Image(mean_prediction.squeeze()) if mean_prediction is not None else None,
            f"{file_name}/std_dev": wandb.Image(std_dev.squeeze()) if std_dev is not None else None,
            f"{file_name}/quantile_spread": wandb.Image(quantile_spread.squeeze()) if quantile_spread is not None else None,
        })


In [ ]:
# Define navigation functions
def next_image(_=None):
    global current_index
    current_index += 1
    visualize_prediction(current_index)

def prev_image(_=None):
    global current_index
    current_index -= 1
    visualize_prediction(current_index)

def go_to_index(change):
    global current_index
    current_index = int(change["new"]) - 1
    visualize_prediction(current_index)

In [ ]:
# Set up interactive widgets
prev_button = widgets.Button(description="⬅️ Previous")
next_button = widgets.Button(description="Next ➡️")
index_slider = widgets.IntSlider(min=1, max=len(predictions), step=1, description="Image:")

prev_button.on_click(prev_image)
next_button.on_click(next_image)
index_slider.observe(go_to_index, names="value")

In [ ]:
# Display UI
display(widgets.HBox([prev_button, index_slider, next_button]))
visualize_prediction(current_index)